In [1]:
!pip install transformers datasets accelerate evaluate -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/HarshAggarwal524/hinglish-sentiment-analysis
%cd /content/hinglish-sentiment-analysis

Cloning into 'hinglish-sentiment-analysis'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 64 (delta 18), reused 55 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 3.18 MiB | 13.79 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/hinglish-sentiment-analysis


In [3]:
import sys
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

sys.path.append("/content/hinglish-sentiment-analysis")

In [4]:
train_custom = pd.read_csv("data/train_custom.csv")
dev_custom = pd.read_csv("data/dev_custom.csv")

label2id = {"negative": 0, "neutral": 1, "positive": 2}
train_custom["label"] = train_custom["sentiment"].map(label2id)
dev_custom["label"] = dev_custom["sentiment"].map(label2id)

print("Train:", len(train_custom), "Dev:", len(dev_custom))
print("Overlap:", len(set(train_custom['tweet_id']) & set(dev_custom['tweet_id'])))

Train: 13158 Dev: 2322
Overlap: 0


In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=200,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
    report_to="none"
)

In [6]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

train_ds = Dataset.from_pandas(train_custom[["text", "label"]]).map(tokenize_function, batched=True)
dev_ds = Dataset.from_pandas(dev_custom[["text", "label"]]).map(tokenize_function, batched=True)
train_ds = train_ds.rename_column("label", "labels")
dev_ds = dev_ds.rename_column("label", "labels")

print("Train:", len(train_ds), "Dev:", len(dev_ds))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/13158 [00:00<?, ? examples/s]

Map:   0%|          | 0/2322 [00:00<?, ? examples/s]

Train: 13158 Dev: 2322


In [7]:
model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=3)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    compute_metrics=compute_metrics
)

trainer.train()

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.909427,0.900781,0.582687,0.587023
2,0.793854,0.831328,0.627476,0.630866
3,0.709093,0.839751,0.638674,0.642047
4,0.639528,0.904731,0.636951,0.638694
5,0.596116,0.919889,0.641688,0.644162


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4115, training_loss=0.7417742042031642, metrics={'train_runtime': 1829.2586, 'train_samples_per_second': 35.965, 'train_steps_per_second': 2.25, 'total_flos': 4327557938081280.0, 'train_loss': 0.7417742042031642, 'epoch': 5.0})

In [8]:
predictions = trainer.predict(dev_ds)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print("XLM-R Classification Report:")
print(classification_report(labels, preds, target_names=['negative', 'neutral', 'positive']))
print("\nConfusion Matrix:")
print(confusion_matrix(labels, preds))

XLM-R Classification Report:
              precision    recall  f1-score   support

    negative       0.64      0.68      0.66       683
     neutral       0.59      0.55      0.57       868
    positive       0.70      0.71      0.70       771

    accuracy                           0.64      2322
   macro avg       0.64      0.65      0.64      2322
weighted avg       0.64      0.64      0.64      2322


Confusion Matrix:
[[466 179  38]
 [191 478 199]
 [ 72 153 546]]


In [9]:
tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

train_ds = Dataset.from_pandas(train_custom[["text", "label"]]).map(tokenize_function, batched=True)
dev_ds = Dataset.from_pandas(dev_custom[["text", "label"]]).map(tokenize_function, batched=True)
train_ds = train_ds.rename_column("label", "labels")
dev_ds = dev_ds.rename_column("label", "labels")

print("Train:", len(train_ds), "Dev:", len(dev_ds))

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Map:   0%|          | 0/13158 [00:00<?, ? examples/s]

Map:   0%|          | 0/2322 [00:00<?, ? examples/s]

Train: 13158 Dev: 2322


In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    "google/muril-base-cased",
    num_labels=3
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    compute_metrics=compute_metrics
)

trainer.train()

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.907140,0.878951,0.596469,0.598935
2,0.763231,0.832449,0.636090,0.639415
3,0.694999,0.831089,0.629630,0.633073
4,0.600603,0.879966,0.637382,0.640647
5,0.564157,0.918687,0.638674,0.641085


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4115, training_loss=0.7279835838883355, metrics={'train_runtime': 2222.6582, 'train_samples_per_second': 29.6, 'train_steps_per_second': 1.851, 'total_flos': 4327557938081280.0, 'train_loss': 0.7279835838883355, 'epoch': 5.0})

In [12]:
predictions = trainer.predict(dev_ds)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print("MuRIL Classification Report:")
print(classification_report(labels, preds, target_names=['negative', 'neutral', 'positive']))
print("\nConfusion Matrix:")


MuRIL Classification Report:
              precision    recall  f1-score   support

    negative       0.67      0.61      0.64       683
     neutral       0.57      0.58      0.57       868
    positive       0.70      0.73      0.71       771

    accuracy                           0.64      2322
   macro avg       0.64      0.64      0.64      2322
weighted avg       0.64      0.64      0.64      2322


Confusion Matrix:


In [13]:
print(confusion_matrix(labels, preds))


[[417 231  35]
 [156 503 209]
 [ 53 155 563]]
